# Nemotron LoRA — train on Kaggle (free 2xT4, 4-bit QLoRA)

**Run this HEADLESS so it survives browser disconnects:** Save Version -> **Save & Run All (Commit)**. It runs in the background; close the tab and come back.

**Before running, Add Input:**
- the **competition data**
- the model **`nemotron-3-nano-30b-a3b-bf16`** (publisher `metric`) — so there's NO 60 GB download; the trainer auto-detects the mount.
- Accelerator: **GPU T4 x2**. Internet: **On** (for pip + the data download fallback).

4-bit NF4 + bf16 compute => ~16 GB weights, fits 2xT4 (32 GB) with room.

## 1. Code + dependencies

In [ ]:
!git clone -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
%cd repo
!pip install -q "transformers>=4.45,<5" peft trl datasets accelerate bitsandbytes psutil einops

## 2. Build mamba_ssm + causal_conv1d from source (matches Kaggle's torch, ~10 min)

In [ ]:
import os
os.environ["CAUSAL_CONV1D_FORCE_BUILD"]="TRUE"; os.environ["MAMBA_FORCE_BUILD"]="TRUE"; os.environ["MAX_JOBS"]="2"
!pip install -q ninja packaging wheel setuptools
!pip install -q --no-build-isolation causal-conv1d
!pip install -q --no-build-isolation mamba-ssm
!python -c "import causal_conv1d, mamba_ssm; print('mamba OK')"

## 3. Competition data (from the attached dataset; recursive find)

In [ ]:
import glob, os, shutil
os.makedirs('data', exist_ok=True)
hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
assert hits, "train.csv not found — Add Input -> the competition"
shutil.copy(hits[0], 'data/train.csv'); print('train.csv <-', hits[0])

## 4. EDA + build the SFT data

In [ ]:
!python scripts/01_eda.py --data-dir data
!python scripts/02_prepare_data.py --data-dir data

## 5. Train (4-bit QLoRA on 2xT4)
Loads the base from the attached model mount (auto-detected). NF4 + bf16 compute. A smoke test runs first. ~1.5-3 h on 2xT4 — well within the 12 h commit limit.

In [ ]:
import os
os.environ['QUANT'] = '4bit'
os.environ['NEMOTRON_MAX_MEMORY_GPU'] = '14GiB'   # per T4
os.environ['SFT_MAX_SEQ_LENGTH'] = '1024'
os.environ['NUM_EPOCHS'] = '2'
!python scripts/03_train_lora.py --data-path data/train_sft.jsonl --output-dir /kaggle/working/lora_adapter

## 6. Package the submission
Writes submission.zip (adapter at zip root, r<=32). After the commit finishes, submit it from the notebook's **Output**, or download `lora_adapter` for the `kaggle_package_submit.ipynb` flow.

In [ ]:
!python scripts/05_package_submission.py --adapter-dir /kaggle/working/lora_adapter --output /kaggle/working/submission.zip